# Fan-out latency — example analysis

One run of the harness: 200k messages at 250k msg/s, mixed types, 1024 slots.

```bash
make -C ../harness
../harness/bin/producer --shm /fanout_demo --slots 1024 --count 0 --rate 250000 --type mixed &
../harness/bin/consumer --shm /fanout_demo --slots 1024 --from-edge \
    --count 200000 --csv data/latency.csv
```

In [ ]:
import numpy as np
import pandas as pd
from bokeh.io import output_notebook, show
from bokeh.models import ColumnDataSource, HoverTool, NumeralTickFormatter
from bokeh.plotting import figure

output_notebook()

BLUE = "#2a78d6"

ns = pd.read_csv("data/latency.csv")["latency_ns"].to_numpy()
ns.size, ns.min(), int(ns.mean()), ns.max()

## Distribution

Clipped at p99.9 for display purposes

In [ ]:
clip = np.percentile(ns, 99.9)
counts, edges = np.histogram(ns[ns <= clip], bins=60)
src = ColumnDataSource(dict(count=counts, left=edges[:-1], right=edges[1:],
                            mid=(edges[:-1] + edges[1:]) / 2))

p = figure(height=320, sizing_mode="stretch_width",
           title=f"Delivery latency, clipped at p99.9 = {clip:,.0f} ns",
           x_axis_label="latency (ns)", y_axis_label="messages",
           tools="pan,box_zoom,reset,save")
p.quad(source=src, bottom=0, top="count", left="left", right="right",
       fill_color=BLUE, line_color="white")
p.y_range.start = 0
p.xaxis.formatter = p.yaxis.formatter = NumeralTickFormatter(format="0,0")
p.add_tools(HoverTool(tooltips=[("latency", "@mid{0,0} ns"),
                                ("messages", "@count{0,0}")], mode="vline"))
show(p)

## Tail

Log axis on both scales — a mean hides the tail entirely.

In [ ]:
# Plotted against "distance from 100th percentile" on a log axis, so the tail gets
# real width instead of being squeezed into the last few pixels.
# Stops at p99.99: beyond that is a handful of samples out of 200k, not a stable figure.
pcts = np.array([50, 75, 90, 99, 99.9, 99.99])
curve = ColumnDataSource(dict(x=100 - pcts, ns=np.percentile(ns, pcts),
                              pct=[f"p{v:g}" for v in pcts]))

q = figure(height=320, sizing_mode="stretch_width",
           x_axis_type="log", y_axis_type="log",
           x_range=(70, 0.0025),  # reversed: p50 on the left, p99.99 on the right
           title="Latency tail",
           x_axis_label="percentile", y_axis_label="latency (ns, log)",
           tools="pan,box_zoom,reset,save")
q.line(source=curve, x="x", y="ns", color=BLUE, width=2)
q.scatter(source=curve, x="x", y="ns", color=BLUE, size=9,
          line_color="white", line_width=2)
q.xaxis.ticker = list(100 - pcts)
q.xaxis.major_label_overrides = {100 - v: f"p{v:g}" for v in pcts}
q.yaxis.ticker = [100, 200, 500, 1000, 2000, 5000]
q.yaxis.formatter = NumeralTickFormatter(format="0,0")
q.add_tools(HoverTool(tooltips=[("percentile", "@pct"), ("latency", "@ns{0,0} ns")]))
show(q)

pd.DataFrame([{f"p{v:g}": int(x) for v, x in zip(pcts, np.percentile(ns, pcts))}])